# SolarScan — 02 · Préparation des données

L'EDA (notebook 01) nous a montré un **fort déséquilibre** des classes. Ici, on transforme les données brutes en **jeux d'entraînement / validation / test** propres et **reproductibles**.

**Pourquoi 3 jeux ?**
- **train** : le modèle apprend dessus.
- **validation** : on règle/surveille le modèle dessus (sans tricher).
- **test** : jugement final, sur des données *jamais vues*.

Et surtout : un découpage **stratifié** pour que les classes rares restent proportionnelles partout.

In [ ]:
# scikit-learn nous donne le decoupage stratifie
%pip install -q scikit-learn pandas

## 0. Imports

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path('../data')

## 1. Construire la table (image → classe)

On met les métadonnées dans un `DataFrame` pandas : une ligne = une image, avec son chemin et sa classe.

In [2]:
with open(DATA_DIR / 'module_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

data = pd.DataFrame([
    {'filepath': v['image_filepath'], 'classe': v['anomaly_class']}
    for v in meta.values()
])
print('Table :', data.shape)
data.head()

Table : (20000, 2)


,filepath,classe
0,images/13357.jpg,No-Anomaly
1,images/13356.jpg,No-Anomaly
2,images/19719.jpg,No-Anomaly
3,images/11542.jpg,No-Anomaly
4,images/11543.jpg,No-Anomaly


## 2. Découpage stratifié train / val / test

On vise **70 % / 15 % / 15 %**. L'option `stratify=` garantit que **chaque classe garde la même proportion** dans les 3 jeux — essentiel vu nos classes rares (Diode-Multi n'a que 175 images !).

On fixe `random_state=42` pour que le découpage soit **toujours identique** (reproductibilité).

In [3]:
# 1er decoupage : 70% train, 30% temporaire
train_df, temp_df = train_test_split(
    data, test_size=0.30, stratify=data['classe'], random_state=42)

# 2e decoupage : on coupe le temporaire en deux -> 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['classe'], random_state=42)

print('train :', len(train_df))
print('val   :', len(val_df))
print('test  :', len(test_df))

train : 14000
val   : 3000
test  : 3000


## 3. Vérifier que les proportions sont préservées

Réflexe DS : on **vérifie** que la stratification a bien marché. Les colonnes `train`, `val`, `test` doivent être quasi identiques à `global` pour chaque classe.

In [4]:
prop = pd.DataFrame({
    'global': data['classe'].value_counts(normalize=True),
    'train':  train_df['classe'].value_counts(normalize=True),
    'val':    val_df['classe'].value_counts(normalize=True),
    'test':   test_df['classe'].value_counts(normalize=True),
}).mul(100).round(1)
prop

,global,train,val,test
classe,,,,
No-Anomaly,50.0,50.0,50.0,50.0
Cell,9.4,9.4,9.4,9.4
Vegetation,8.2,8.2,8.2,8.2
Diode,7.5,7.5,7.5,7.5
Cell-Multi,6.4,6.4,6.4,6.4
Shadowing,5.3,5.3,5.3,5.3
Cracking,4.7,4.7,4.7,4.7
Offline-Module,4.1,4.1,4.1,4.1
Hot-Spot,1.2,1.2,1.3,1.2


## 4. Sauvegarder les splits (reproductibilité)

On fige les 3 jeux dans des fichiers CSV. Ainsi, le notebook de modélisation utilisera **exactement** le même découpage — indispensable pour comparer des modèles équitablement.

In [5]:
out = Path('../splits')
out.mkdir(exist_ok=True)
train_df.to_csv(out / 'train.csv', index=False)
val_df.to_csv(out / 'val.csv', index=False)
test_df.to_csv(out / 'test.csv', index=False)
print('Splits sauvegardes dans :', out.resolve())
print(sorted(p.name for p in out.glob('*.csv')))

Splits sauvegardes dans : C:\Users\HP PRO\Desktop\souley\projets\SolarScan\splits
['test.csv', 'train.csv', 'val.csv']


## 5. Bilan

- Découpage **stratifié 70 / 15 / 15**, reproductible (`random_state=42`).
- Proportions de classes **préservées** dans chaque jeu (vérifié en section 3).
- Splits **sauvegardés** dans `splits/` (réutilisables et versionnables).

➡️ Prochain notebook : **modélisation** — transfer learning avec ResNet-18, en gérant le déséquilibre (pondération des classes) et en suivant le **macro-F1**.